### Loading data in hierarchal HDF5 format

Subject -> Stimulus Modality -> Stimulus Font -> Parity or Control

In [ ]:
import os
import pandas as pd
import mne
from pr_fe import FeatureExtractor
import h5py
import numpy as np

In [ ]:
input_list_path = 'data/mat_files_cleaned.txt'
data_dir = 'data'
output_base_dir = 'hierarch_gr'

fe = FeatureExtractor()

In [ ]:
def parse_filename(filename):
    base = os.path.basename(filename).replace('.mat', '')
    parts = base.split('_')

    subject = next((p[1:] for p in reversed(parts) if p.startswith('S')), 'Unknown')

    try:
        idx = parts.index('epbin') + 1
    except ValueError:
        idx = 1

    modality = parts[idx] if len(parts) > idx else 'UnknownModality'
    font = parts[idx + 1] if len(parts) > idx + 1 else 'UnknownFont'
    condition = parts[idx + 2] if len(parts) > idx + 2 else 'UnknownCondition'

    return subject, modality, font, condition

In [ ]:
def load_and_prepare_csv(filepath):
    df = pd.read_csv(filepath)
    
    df_cond0 = df[df['condition'] == 0].pivot(index='time', columns='channel', values='value')
    df_cond1 = df[df['condition'] == 1].pivot(index='time', columns='channel', values='value')
    
    df_cond0.columns = df_cond0.columns.astype(str)
    df_cond1.columns = df_cond1.columns.astype(str)
    common_channels = sorted(list(set(df_cond0.columns) & set(df_cond1.columns)))
    df_cond0 = df_cond0[common_channels]
    df_cond1 = df_cond1[common_channels]
    
    return df_cond0, df_cond1, common_channels

In [ ]:
def process_condition(df_signal, channels, condition_label):
    ch_types = ['eeg'] * len(channels)
    sfreq = fe.sampling_rate
    data = df_signal[channels].T.values 

    info = mne.create_info(ch_names=channels, sfreq=sfreq, ch_types=ch_types)
    raw = mne.io.RawArray(data, info)

    features_df = fe.merging_feature_data(raw, df_signal)
    
    features_df = features_df.apply(pd.to_numeric, errors='coerce')
    features_df = features_df.select_dtypes(include=[np.number])
    features_df = features_df.dropna(axis=1, how='all')
    
    return features_df

In [ ]:
with open(input_list_path, 'r') as f:
        files = [line.strip() for line in f if line.strip()]

for file_rel_path in files:
    csv_rel_path = file_rel_path.replace('.mat', '.csv')
    file_path = os.path.join(data_dir, csv_rel_path)

    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}, skipping.")
        continue

    print(f"Processing {file_path}...")

    subject, modality, font, condition = parse_filename(file_rel_path)
    df_cond0, df_cond1, common_channels = load_and_prepare_csv(file_path)

    for cond, df in [('0', df_cond0), ('1', df_cond1)]:
        try:
            features_df = process_condition(df, common_channels, cond)
            
            output_dir = os.path.join(output_base_dir, f'S{subject}', modality, font)
            os.makedirs(output_dir, exist_ok=True)
            output_file = os.path.join(output_dir, f'{condition}_{cond}.h5')
            group_name = f'S{subject}/{modality}/{font}'

            with h5py.File(output_file, 'a') as hdf5_file:
                group = hdf5_file.require_group(group_name)
                dataset_name = f'{condition}_{cond}'
                
                if dataset_name in group:
                    print(f"Dataset {dataset_name} exists, skipping.")
                    continue

                group.create_dataset(dataset_name, data=features_df.to_numpy(), chunks=True)
                group.attrs['columns'] = np.array(features_df.columns, dtype='S')
                group.attrs['condition'] = int(cond)

            print(f"Saved condition {cond} features to {output_file}")
            
        except Exception as e:
            print(f"Failed to process condition {cond}: {str(e)}")